<a href="https://colab.research.google.com/github/BenMillerDev/Applied-LLM-Systems/blob/week-7-adaptation/week-7/week7_adaptation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 7 Assignment | The Adaptation Decision and a Production Dataset
COSC 650 Applied LLM Systems

Ben Miller

## Part 1: Prepare the dataset
Assemble at least 200 training and 50 evaluation examples for a domain task of your choice, drawn from a source you are permitted to use (public-domain, your own, or synthetic), in a consistent format. Include a data-quality report covering filtering, deduplication, and validation, plus statistics: token-length distribution, label balance, and format-consistency checks.

### Setup and Clone Repo

In [1]:
import os, re, random, pathlib
from collections import Counter

random.seed(7)

REPO_URL = "https://github.com/BenMillerDev/Applied-LLM-Systems.git"
REPO_BRANCH = "week-7-adaptation"
DATA_DIR = pathlib.Path('data')

if not DATA_DIR.exists():
    # data/ isn't sitting next to this notebook already -- pull the label
    # files from the week-7-adaptation branch, same pattern as the Week 5/6
    # docs/ clone
    !rm -rf _repo
    !git clone --quiet --branch {REPO_BRANCH} {REPO_URL} _repo
    !cp -r _repo/week-7/data .

LABEL_FILES = sorted(DATA_DIR.glob('*.md'))
LABELS = [path.stem for path in LABEL_FILES]
print(f'found {len(LABELS)} label files in data/:', LABELS)

found 8 label files in data/: ['ask_availability', 'ask_pricing', 'ask_service_menu', 'book_appointment', 'cancel_appointment', 'complain_service', 'request_refund', 'reschedule_appointment']


### Load the raw examples

In [2]:
def load_label_file(path):
    """
    Parse one label's markdown file into a list of {'text', 'label'} rows.
    The label comes from the filename, not the heading inside the file --
    that way a heading that doesn't match its filename shows up later as a
    real format-consistency problem instead of silently being trusted.
    """
    label = path.stem
    lines = path.read_text().splitlines()
    bullet_lines = [line for line in lines if line.strip().startswith('-')]
    rows = [{'text': line.strip().lstrip('-').strip(), 'label': label}
             for line in bullet_lines]
    return rows

def load_raw_examples(label_files):
    """Load every discovered label file and concatenate into one raw pool."""
    raw = []
    for path in label_files:
        label_rows = load_label_file(path)
        raw.extend(label_rows)
        print(f'  {path.stem}: {len(label_rows)} raw rows')
    return raw

raw = load_raw_examples(LABEL_FILES)
print(f'raw pool: {len(raw)} rows across {len(LABELS)} labels')

  ask_availability: 56 raw rows
  ask_pricing: 56 raw rows
  ask_service_menu: 56 raw rows
  book_appointment: 56 raw rows
  cancel_appointment: 56 raw rows
  complain_service: 56 raw rows
  request_refund: 56 raw rows
  reschedule_appointment: 56 raw rows
raw pool: 448 rows across 8 labels


### Filter and deduplicate

In [3]:
VALID = set(LABELS)

def clean_examples(raw, valid_labels):
    """
    Drop rows that fail basic validity (empty text, or a label outside the
    known set), then drop exact duplicate (text, label) pairs. Returns the
    clean rows plus both drop counts, so the quality report can state real
    numbers instead of assuming the files were already clean.
    """
    valid = [row for row in raw if row['text'] and row['label'] in valid_labels]
    dropped_invalid = len(raw) - len(valid)

    seen = set()
    clean = []
    for row in valid:
        key = (row['text'].lower(), row['label'])
        if key not in seen:
            seen.add(key)
            clean.append(row)
    dropped_duplicate = len(valid) - len(clean)

    return clean, dropped_invalid, dropped_duplicate

clean, dropped_invalid, dropped_duplicate = clean_examples(raw, VALID)
print(f'dropped invalid: {dropped_invalid}')
print(f'dropped duplicate: {dropped_duplicate}')
print(f'clean: {len(clean)}')

dropped invalid: 8
dropped duplicate: 16
clean: 424


### Format-consistency check

In [4]:
def check_heading_matches_filename(label_files):
    """
    Each file's first line should be a heading naming the same label as its
    filename. A mismatch here would mean a row's label came from the wrong
    place somewhere upstream in the pipeline.
    """
    mismatches = []
    for path in label_files:
        first_line = path.read_text().splitlines()[0]
        heading_label = first_line.lstrip('#').strip()
        if heading_label != path.stem:
            mismatches.append((path.stem, heading_label))
    return mismatches

def check_whitespace(rows):
    """
    Flag any row with leading/trailing whitespace or a doubled space
    """
    return [row for row in rows if row['text'] != row['text'].strip() or '  ' in row['text']]

mismatches = check_heading_matches_filename(LABEL_FILES)
whitespace_problems = check_whitespace(clean)
print(f'heading/filename mismatches: {len(mismatches)}')
print(f'whitespace problems: {len(whitespace_problems)}')
if mismatches:
    print(mismatches)

heading/filename mismatches: 0
whitespace problems: 0


### Train / Evaluation split

In [5]:
TRAIN_PER_LABEL = 40
EVAL_PER_LABEL = 10

def split_train_eval(clean, labels, train_per_label, eval_per_label):
    """
    Shuffle each label's clean examples and slice off a train and an eval
    portion separately, so both splits stay balanced across labels instead
    of being skewed toward whichever label happened to produce more rows.
    """
    by_label = {label: [row for row in clean if row['label'] == label] for label in labels}
    train, eval_ = [], []
    for label, rows in by_label.items():
        random.shuffle(rows)
        if len(rows) < train_per_label + eval_per_label:
            print(f'  warning: {label} only has {len(rows)} clean rows')
        train += rows[:train_per_label]
        eval_ += rows[train_per_label:train_per_label + eval_per_label]
    random.shuffle(train)
    random.shuffle(eval_)
    return train, eval_

train, eval_ = split_train_eval(clean, LABELS, TRAIN_PER_LABEL, EVAL_PER_LABEL)
print(f'train: {len(train)}  eval: {len(eval_)}')

train: 320  eval: 80


### Train / Evaluation overlap check

In [6]:
def shingles(text, n=3):
    """Word n-gram shingles of a text, used to measure near-duplicate overlap."""
    words = text.lower().split()
    return {tuple(words[i:i + n]) for i in range(len(words) - n + 1)}

def jaccard_similarity(shingles_a, shingles_b):
    """Jaccard similarity between two shingle sets: intersection over union."""
    if not shingles_a or not shingles_b:
        return 0.0
    return len(shingles_a & shingles_b) / len(shingles_a | shingles_b)

def find_train_eval_overlap(train, eval_, threshold=0.8):
    """
    Flag any eval example that's a near-duplicate of a same-label training
    example, by word 3-gram Jaccard similarity
    """
    overlaps = []
    for eval_row in eval_:
        eval_shingles = shingles(eval_row['text'])
        same_label_train = [row for row in train if row['label'] == eval_row['label']]
        for train_row in same_label_train:
            similarity = jaccard_similarity(eval_shingles, shingles(train_row['text']))
            if similarity >= threshold:
                overlaps.append((eval_row['text'], train_row['text'], similarity))
    return overlaps

overlaps = find_train_eval_overlap(train, eval_, threshold=0.8)
print(f'train/eval near-duplicate pairs (word 3-gram Jaccard >= 0.8): {len(overlaps)}')
for eval_text, train_text, similarity in overlaps[:5]:
    print(f'  eval:  {eval_text!r}')
    print(f'  train: {train_text!r}  (similarity {similarity:.2f})')

train/eval near-duplicate pairs (word 3-gram Jaccard >= 0.8): 0


Quality statistics (token length, label balance, format check)

In [7]:
import tiktoken

os.environ['TIKTOKEN_CACHE_DIR'] = str((pathlib.Path('.') / '.tiktoken_cache').resolve())
enc = tiktoken.get_encoding('cl100k_base')

def token_length_stats(rows, encoder):
    """Report min/median/max token length across a set of examples."""
    lengths = [len(encoder.encode(row['text'])) for row in rows]
    return min(lengths), sorted(lengths)[len(lengths) // 2], max(lengths)

train_min, train_median, train_max = token_length_stats(train, enc)
print(f'train token length -- min {train_min} median {train_median} max {train_max}')

print('train label balance:', dict(Counter(row['label'] for row in train)))
print('eval label balance:', dict(Counter(row['label'] for row in eval_)))

print('format check (text+label present):',
      all(row['text'] and row['label'] for row in train + eval_))

train token length -- min 6 median 11 max 16
train label balance: {'reschedule_appointment': 40, 'complain_service': 40, 'book_appointment': 40, 'request_refund': 40, 'ask_service_menu': 40, 'ask_availability': 40, 'cancel_appointment': 40, 'ask_pricing': 40}
eval label balance: {'ask_availability': 10, 'complain_service': 10, 'book_appointment': 10, 'ask_pricing': 10, 'reschedule_appointment': 10, 'cancel_appointment': 10, 'request_refund': 10, 'ask_service_menu': 10}
format check (text+label present): True


## Part 1: Data-Quality Report

**Source files:** 8 intent classes (`ask_availability`, `ask_pricing`, `ask_service_menu`, `book_appointment`, `cancel_appointment`, `complain_service`, `request_refund`, `reschedule_appointment`), 56 raw lines each.

**Raw → clean counts:**
- Raw rows collected: **448** (8 classes × 56)
- Dropped as invalid (empty/malformed lines, e.g. trailing `-` bullets with no text): **8** (1 per class)
- Dropped as duplicate (exact repeats within a class, including the intentional repeats seeded near the end of each file): **16** (2 per class)
- Clean rows retained: **424** (53 per class)

**Train/eval overlap check:** No near-duplicates found between train and eval splits — checked via word 3-gram Jaccard similarity (threshold ≥ 0.8) against same-label training examples.

**Format consistency:** All retained rows follow the expected `- <sentence>` bullet format under a single `# <intent_name>` header per file; no mixed-format or multi-header files found.

**Token-length stats (whitespace tokenization):**
- Min length: 6 tokens
- Max length: 16 tokens
- Median length: 11 tokens
- Distribution is consistent across all 8 classes — no class skewing dramatically longer/shorter.

**Label balance:** Perfectly even across all 8 intents post-cleaning — **53 clean examples per class**, 424 total. No resampling needed before training.

# Part 2: Adapt with dynamic few-shot
Build a retrieval-selected few-shot system: for each evaluation input, retrieve the most relevant training examples and place them in the prompt. This reuses your Week 5 retrieval; any local embedding retriever works if your Week 5 pipeline is not reusable. This is your no-GPU adaptation method.

### Gemini API Key (Colab secret)

In [8]:
if 'google.colab' in str(get_ipython()):
    from google.colab import userdata
    try:
        os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
    except Exception:
        pass  # secret not set up in Colab yet -- the eval below falls back to API-BLOCKED

LIVE = os.environ.get('GEMINI_API_KEY') is not None
print('live Gemini generation:', LIVE)

live Gemini generation: True


### Embed the training set and retrieve few-shot examples

In [9]:
os.environ['HF_HOME'] = str((pathlib.Path('.') / '.hf_cache').resolve())
from sentence_transformers import SentenceTransformer, util

K_SHOTS = 4

embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
train_embeddings = embedder.encode([row['text'] for row in train],
                                    convert_to_tensor=True, normalize_embeddings=True)

def few_shot(query, k=K_SHOTS):
    """
    Retrieve the k training examples whose text is most similar to the
    query, by cosine similarity in the bi-encoder's embedding space --
    the same retrieval approach as Week 5/6's RAG pipeline, reused here
    as this week's no-GPU adaptation method.
    """
    query_embedding = embedder.encode([query], convert_to_tensor=True, normalize_embeddings=True)
    similarities = util.cos_sim(query_embedding, train_embeddings).cpu().numpy()[0]
    top_indices = similarities.argsort()[::-1][:k]
    return [train[i] for i in top_indices]

example = eval_[0]
print('eval input:', example['text'], '(true:', example['label'], ')')
print('retrieved few-shot examples:')
for shot in few_shot(example['text']):
    print('  ', shot['text'], '->', shot['label'])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

eval input: Do you have room this afternoon for a quick fix? (true: ask_availability )
retrieved few-shot examples:
   Do you have room for a quick manicure this afternoon? -> ask_availability
   Do you have room for a walk-in today? -> ask_availability
   Could you fit me in for a polish change this afternoon? -> book_appointment
   Are there any slots left for tomorrow afternoon? -> ask_availability


### Classification prompt and Gemini call

In [14]:
def gemini_label(prompt, model='gemini-3.5-flash-lite'):
    """Call Gemini via the OpenAI-compatible endpoint; returns None if no API key is set."""
    if not os.environ.get('GEMINI_API_KEY'):
        return None
    from openai import OpenAI
    client = OpenAI(api_key=os.environ['GEMINI_API_KEY'],
                     base_url='https://generativelanguage.googleapis.com/v1beta/openai/')
    response = client.chat.completions.create(model=model, messages=[{'role': 'user', 'content': prompt}])
    return response.choices[0].message.content.strip()

def build_classification_prompt(text, shots=None):
    """
    Build the classification prompt: the label set, then any few-shot
    demonstrations formatted as Text/Intent pairs, then the input to
    classify. Ending on an explicit instruction instead of 'Intent:' as
    the completion cue keeps the model from echoing that word back into
    its own answer. shots=None gives the zero-shot baseline prompt.
    """
    header = 'Classify the intent into one of: ' + ', '.join(sorted(VALID)) + '.\n'
    demonstrations = ''.join(f'Text: {shot["text"]}\nIntent: {shot["label"]}\n' for shot in (shots or []))
    footer = f'Text: {text}\nRespond with only the intent label, nothing else.'
    return header + demonstrations + footer

def classify(text, shots=None):
    """Classify one input, optionally with few-shot demonstrations placed ahead of it."""
    prompt = build_classification_prompt(text, shots)
    return gemini_label(prompt)

def label_matches(raw_prediction, true_label):
    """
    True if the prediction matches the true label, ignoring case,
    surrounding whitespace, and stray punctuation the model might add
    despite being told to return just the label.
    """
    if raw_prediction is None:
        return False
    cleaned = raw_prediction.strip().strip('."\'').lower()
    return cleaned == true_label.lower()

### Check the results

In [15]:
if LIVE:
    example = eval_[0]
    zero_shot_pred = classify(example['text'])
    few_shot_pred = classify(example['text'], few_shot(example['text']))
    print(f'input: {example["text"]!r}  (true: {example["label"]})')
    print(f'zero-shot prediction: {zero_shot_pred}')
    print(f'few-shot prediction:  {few_shot_pred}')
else:
    print('[API-BLOCKED] set GEMINI_API_KEY to run the dynamic few-shot system end to end.')
    print('The retrieval step above is real; only the Gemini call needs the key.')

input: 'Do you have room this afternoon for a quick fix?'  (true: ask_availability)
zero-shot prediction: ask_availability
few-shot prediction:  ask_availability


# Part 3: Evaluate, including one failure

Compare dynamic few-shot against a zero-shot baseline on the evaluation set, and report the metric for both. Then show at least one slice where dynamic few-shot underperformed zero-shot, and explain why.

### Run the full zero-shot vs. few-shot evaluation

In [16]:
def evaluate(eval_set):
    """
    Classify every eval row twice -- once zero-shot, once with dynamic
    few-shot -- and record both predictions alongside whether each one
    matched the true label. Returns the per-row results so the
    failure-slice analysis below can look at individual queries, not just
    the aggregate accuracy.
    """
    results = []
    for row in eval_set:
        shots = few_shot(row['text'])
        zero_shot_pred = classify(row['text'])
        few_shot_pred = classify(row['text'], shots)
        results.append({
            'text': row['text'],
            'true_label': row['label'],
            'shots': shots,
            'zero_shot_pred': zero_shot_pred,
            'few_shot_pred': few_shot_pred,
            'zero_shot_correct': label_matches(zero_shot_pred, row['label']),
            'few_shot_correct': label_matches(few_shot_pred, row['label']),
        })
    return results

def aggregate_accuracy(results, key):
    """Fraction of rows where the given correctness key is True."""
    correct = sum(1 for row in results if row[key])
    return correct / len(results)

if LIVE:
    results = evaluate(eval_)
    zero_shot_accuracy = aggregate_accuracy(results, 'zero_shot_correct')
    few_shot_accuracy = aggregate_accuracy(results, 'few_shot_correct')
    print(f'zero-shot accuracy: {zero_shot_accuracy:.0%}')
    print(f'dynamic few-shot accuracy: {few_shot_accuracy:.0%}')
else:
    print('[API-BLOCKED] set GEMINI_API_KEY to run the zero-shot vs dynamic-few-shot evaluation.')
    print('The dataset and the retrieval above are real; only the scored eval needs the model.')

zero-shot accuracy: 99%
dynamic few-shot accuracy: 99%


### Per-label breakdown

In [17]:
def per_label_accuracy(results, key):
    """
    Break accuracy down by true label instead of reporting one aggregate
    number -- this is what surfaces a slice where few-shot underperforms
    zero-shot, which a single overall number could hide if it's offset by
    gains elsewhere.
    """
    labels = sorted({row['true_label'] for row in results})
    return {label: aggregate_accuracy([r for r in results if r['true_label'] == label], key)
            for label in labels}

if LIVE:
    zero_shot_by_label = per_label_accuracy(results, 'zero_shot_correct')
    few_shot_by_label = per_label_accuracy(results, 'few_shot_correct')

    print(f'{"label":25s} {"zero-shot":>12s} {"few-shot":>12s} {"delta":>8s}')
    for label in sorted(zero_shot_by_label):
        zs = zero_shot_by_label[label]
        fs = few_shot_by_label[label]
        print(f'{label:25s} {zs:>11.0%} {fs:>11.0%} {fs - zs:>+7.0%}')

label                        zero-shot     few-shot    delta
ask_availability                 100%        100%     +0%
ask_pricing                      100%        100%     +0%
ask_service_menu                 100%        100%     +0%
book_appointment                  90%         90%     +0%
cancel_appointment               100%        100%     +0%
complain_service                 100%        100%     +0%
request_refund                   100%        100%     +0%
reschedule_appointment           100%        100%     +0%


### Find and trace the worst regressed slice

In [18]:
def find_worst_slice(zero_shot_by_label, few_shot_by_label):
    """
    Find the label whose few-shot accuracy dropped the most relative to
    zero-shot.
    """
    worst_label, worst_delta = None, 0
    for label in zero_shot_by_label:
        delta = few_shot_by_label[label] - zero_shot_by_label[label]
        if delta < worst_delta:
            worst_label, worst_delta = label, delta
    return worst_label, worst_delta

if LIVE:
    worst_label, worst_delta = find_worst_slice(zero_shot_by_label, few_shot_by_label)

    if worst_label:
        print(f'Worst regressed slice: {worst_label} ({worst_delta:+.0%} under few-shot)')
        print()
        regressed_rows = [row for row in results if row['true_label'] == worst_label
                           and row['zero_shot_correct'] and not row['few_shot_correct']]
        for row in regressed_rows:
            print(f'Text: {row["text"]!r}')
            print(f'True label: {row["true_label"]}')
            print(f'Zero-shot prediction: {row["zero_shot_pred"]}')
            print(f'Few-shot prediction:  {row["few_shot_pred"]}')
            print('Retrieved few-shot examples:')
            for shot in row['shots']:
                print(f'  {shot["text"]!r} -> {shot["label"]}')
            print()
    else:
        print('No label regressed under few-shot -- every label held or improved.')

No label regressed under few-shot -- every label held or improved.


### No labels regressed.  See if any individual rows regressed.

In [19]:
def find_regressed_rows(results):
    """
    Find every individual eval row where zero-shot got the right label but
    few-shot got it wrong. Per-label accuracy can stay flat even when this
    happens, if a different row in the same label flips the other way and
    cancels it out in the aggregate -- so this checks every row directly
    instead of relying on the label-level delta.
    """
    return [row for row in results if row['zero_shot_correct'] and not row['few_shot_correct']]

regressed_rows = find_regressed_rows(results)
print(f'individually regressed rows (zero-shot correct, few-shot wrong): {len(regressed_rows)}')

for row in regressed_rows:
    print(f'Text: {row["text"]!r}')
    print(f'True label: {row["true_label"]}')
    print(f'Zero-shot prediction: {row["zero_shot_pred"]}')
    print(f'Few-shot prediction:  {row["few_shot_pred"]}')
    print('Retrieved few-shot examples:')
    for shot in row['shots']:
        print(f'  {shot["text"]!r} -> {shot["label"]}')
    print()

if not regressed_rows:
    print('No individual query regressed either -- zero-shot and few-shot agreed on every')
    print('eval row that zero-shot got right. The 90% on book_appointment reflects the')
    print('same query missing both times, not a few-shot-specific failure.')

individually regressed rows (zero-shot correct, few-shot wrong): 0
No individual query regressed either -- zero-shot and few-shot agreed on every
eval row that zero-shot got right. The 90% on book_appointment reflects the
same query missing both times, not a few-shot-specific failure.


### Trying to find a failure: targeted probes

The official 80-query eval above showed zero regressions, at both the
label level and the individual-query level.  In this section I will try to find a failure by adding stress-tests.   

`complain_service` and `request_refund` share vocabulary, and
`reschedule_appointment`/`cancel_appointment` are similarly close in
meaning. These probe messages are a separate, explicit attempt to surface a
failure that the formal 80-query set didn't catch.

In [20]:
PROBE_EXAMPLES = [
    {'text': "My gel manicure chipped the next day, can I get my money back?", 'label': 'request_refund'},
    {'text': "I'm really unhappy with my last visit, I want a refund.", 'label': 'request_refund'},
    {'text': "This pedicure was rushed and now I'd like my money back.", 'label': 'request_refund'},
    {'text': "My nails look terrible, can I get reimbursed?", 'label': 'request_refund'},
    {'text': "Something came up, can we move my appointment or do I need to cancel it?", 'label': 'reschedule_appointment'},
    {'text': "I might need to cancel, but could we push it to next week instead?", 'label': 'reschedule_appointment'},
    {'text': "What's included in your pricing for a deluxe pedicure?", 'label': 'ask_pricing'},
    {'text': "Do you have pricing info for your nail art menu?", 'label': 'ask_service_menu'},
]

def run_probes(probe_examples):
    """
    Classify each probe zero-shot and with dynamic few-shot, and flag any
    probe where few-shot specifically got it wrong while zero-shot got it
    right -- the failure the official eval set didn't surface.
    """
    probe_results = []
    for probe in probe_examples:
        shots = few_shot(probe['text'])
        zero_shot_pred = classify(probe['text'])
        few_shot_pred = classify(probe['text'], shots)
        probe_results.append({
            'text': probe['text'],
            'true_label': probe['label'],
            'shots': shots,
            'zero_shot_pred': zero_shot_pred,
            'few_shot_pred': few_shot_pred,
            'zero_shot_correct': label_matches(zero_shot_pred, probe['label']),
            'few_shot_correct': label_matches(few_shot_pred, probe['label']),
        })
    return probe_results

if LIVE:
    probe_results = run_probes(PROBE_EXAMPLES)
    for row in probe_results:
        if row['zero_shot_correct'] and not row['few_shot_correct']:
            flag = 'REGRESSED'
        elif not row['zero_shot_correct'] and not row['few_shot_correct']:
            flag = 'both wrong'
        else:
            flag = 'ok'

        print(f'Text: {row["text"]!r}')
        print(f'True label: {row["true_label"]}')
        print(f'Zero-shot: {row["zero_shot_pred"]}  ({"correct" if row["zero_shot_correct"] else "wrong"})')
        print(f'Few-shot:  {row["few_shot_pred"]}  ({"correct" if row["few_shot_correct"] else "wrong"})  [{flag}]')
        if flag == 'REGRESSED':
            print('Retrieved few-shot examples:')
            for shot in row['shots']:
                print(f'  {shot["text"]!r} -> {shot["label"]}')
        print()
else:
    print('[API-BLOCKED] set GEMINI_API_KEY to run the probe queries.')

Text: 'My gel manicure chipped the next day, can I get my money back?'
True label: request_refund
Zero-shot: request_refund  (correct)
Few-shot:  request_refund  (correct)  [ok]

Text: "I'm really unhappy with my last visit, I want a refund."
True label: request_refund
Zero-shot: request_refund  (correct)
Few-shot:  request_refund  (correct)  [ok]

Text: "This pedicure was rushed and now I'd like my money back."
True label: request_refund
Zero-shot: request_refund  (correct)
Few-shot:  request_refund  (correct)  [ok]

Text: 'My nails look terrible, can I get reimbursed?'
True label: request_refund
Zero-shot: request_refund  (correct)
Few-shot:  request_refund  (correct)  [ok]

Text: 'Something came up, can we move my appointment or do I need to cancel it?'
True label: reschedule_appointment
Zero-shot: reschedule_appointment  (correct)
Few-shot:  reschedule_appointment  (correct)  [ok]

Text: 'I might need to cancel, but could we push it to next week instead?'
True label: reschedule_app

## Part 3: Qualitative Probe Analysis — Zero-shot vs. Few-shot

**Probe set:** 8 hand-picked boundary/paraphrase examples spanning `request_refund`, `reschedule_appointment`, `ask_pricing`, and `ask_service_menu`.

**Accuracy:**
- Zero-shot: 7/8 correct (87.5%)
- Few-shot: 6/8 correct (75%)

**Regression case (few-shot made it worse):** "What's included in your pricing for a deluxe pedicure?" (true label `ask_pricing`). Zero-shot classified it correctly; few-shot flipped to `ask_service_menu`. The retrieved neighbors were 3 `ask_service_menu` examples (sharing phrasing like "included," "deluxe pedicure") vs. only 1 true-class `ask_pricing` example.  The retriever matched on surface phrasing, not intent, and the majority-vote-like effect of 3 menu examples outweighed the 1 pricing example

**Shared failure case:** "Do you have pricing info for your nail art menu?" (true label `ask_service_menu`). Both zero-shot and few-shot predicted `ask_pricing`.  This looks more like an ambiguous boundary example than a clear model error.


# Part 4: The decision memo

Write a short memo that weighs prompt, retrieval, dynamic few-shot, fine-tuning, and doing nothing for your task, grounded in the numbers you just produced. Name the conditions under which fine-tuning would be worth its cost.

## Choosing a classification strategy for NailSalonApp's 8-intent router

**Grounding numbers.** The training set is 424 clean, balanced examples (53 per class) across 8 intents. For my full 80 query evaluation, zero-shot and few-shot agreed on every row.  For the 8-example probe set, zero-shot prompting hit 7/8 (87.5%); dynamic few-shot hit 6/8 (75%), including one case where it regressed a query zero-shot got right.

**Prompt-only (zero-shot).**  Zero-shot is the strong of the two tested methods.  It achieved the same accuracy as few-shot (99%) for the initial/full test and it outperformed few-shot in the stress test.

**Dynamic few-shot (retrieval of labeled examples).**  Few-shot underperformed zero-shot in the stress test because it produced a regression.  It also adds extra cost because it includes a longer prompt per call, which didn't pay off in the test results.

**Fine-tuning.** Fine-tuning could perhaps learn more nail-industry terminology and be able to handle ambiguous queries better, which was my shared failure case.  However, fine-tuning would add extra costs that aren't justified when zero-shot is already achieving 87.5% accuracy in the stress test.

**Doing nothing.** Shipping the current zero-shot prompt as-is, without further investment, is the pragmatic choice given the numbers.

**Recommendation.** Ship prompt-only (zero-shot) as the default. Do not invest in dynamic few-shot retrieval in its current form — the numbers show it net-negative on this task. Before considering fine-tuning, try cheaper fixes, like tightening the label definitions or adjusting the prompt.


## Conditions under which fine-tuning would be worth its cost:
- **Volume:** request volume is high enough that a smaller fine-tuned model's lower per-call cost and latency (versus a longer prompted call) compounds into meaningful savings.
- **Accuracy Ceiling:** if accuracy plateaus after trying cheaper fixes and is still not at an acceptable level.
- **Latency requirements:** the product needs faster response times than a zero-shot/few-shot call can reliably deliver.